# PAYBACK Lightweight Assistant — Demo Notebook

This notebook walks through the assistant's capabilities with seven carefully chosen
queries that together exercise every component of the system. Read top-to-bottom.

## What this system is

A multilingual (German/English) shopping assistant for the PAYBACK app. Takes a
natural-language query, classifies intent, optionally expands basket-style queries,
retrieves products from three partner catalogs (dm, EDEKA, Amazon), applies
loyalty-aware ranking, and returns either recommendations, a clarifying question,
or a navigation target.

## Architecture (top to bottom)

```
User query (EN/DE)
       ↓
Intent Agent (Claude tool-use)
       ↓
       ├── navigational + partner       → return navigation target
       ├── support                      → return out-of-scope clarification
       ├── navigational without partner → return partner picker
       ├── specific + high-confidence:
       │     ├── basket query? → Query Expander (catalog-grounded)
       │     │                    → multi-query retrieval with relevance filter
       │     └── single query? → single retrieval
       │     → Loyalty Ranker → return recommendations
       └── vague or low-confidence     → catalog-grounded clarification
```

## What you'll see in this demo

| # | Query | What it demonstrates |
|---|---|---|
| 1 | `wireless mouse` | Basic specific search, single-item retrieval |
| 2 | `something for my dog` | Vague query → catalog-grounded clarification |
| 3 | `essentials for moving into a new house` | Basket query → expansion + dropped-query transparency |
| 4 | Same as 3, with `user_edeka_heavy` | Loyalty signal — diversity bonus surfaces under-used partners |
| 5 | `Windeln bei dm` | German cross-lingual + partner-mention extraction |
| 6 | `take me to the shop` | Navigational without partner → fast hardcoded clarification |
| 7 | `how do I redeem my points` | Support intent → graceful out-of-scope |

In [ ]:
import sys, os, pathlib
# Ensure repo root is on sys.path regardless of CWD
_repo = pathlib.Path(os.getcwd())
if not (_repo / 'app').exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))
os.chdir(_repo)

import asyncio
import json
from app.agents.router import get_default_router
from app.models.schemas import UserContext, Partner

router = get_default_router()

def pretty(response):
    """Print key fields then format recommendations/clarification/navigation."""
    intent = response.intent_result
    print(f"Response type:  {response.response_type}")
    print(f"Language:       {intent.language.value}")
    print(f"Intent:         {intent.intent.value}")
    print(f"Specificity:    {intent.specificity.value} (confidence {intent.confidence:.2f})")
    print(f"Target partner: {intent.target_partner}")
    print(f"Is basket:      {intent.is_basket_query}")
    print(f"Latency:        {response.latency_ms:.0f} ms")
    print(f"Cost:           €{response.estimated_cost_eur:.5f}")

    if response.response_type == "recommendations":
        print(f"\nTop {min(5, len(response.recommendations))} recommendations:")
        for rec in response.recommendations[:5]:
            print(f"  {rec.rank}. [{rec.product.partner.value:6}] "
                  f"{rec.product.name[:55]:55s} "
                  f"(sem={rec.semantic_score:.2f}, "
                  f"loy={rec.loyalty_boost:.2f}, "
                  f"div={rec.diversity_bonus:.2f}, "
                  f"final={rec.final_score:.2f})")
        if response.debug_expanded_queries:
            print(f"\nExpansion proposed: {response.debug_expanded_queries}")
        if response.debug_dropped_queries:
            print(f"Expansion dropped:  {response.debug_dropped_queries}")
    elif response.response_type == "clarification":
        print(f"\nClarifying question: {response.clarification.question}")
        print(f"Options: {response.clarification.suggested_options}")
    elif response.response_type == "navigation":
        print(f"\nNavigate to: {response.navigation_target}")

print("✓ Router initialized and ready")

## Query 1 — Basic specific search

Single-item English query. Expectation: retrieval returns Amazon electronics products.
Single-query path (no expansion). Cold-start ranking (no user_id).

In [ ]:
response = await router.handle(query="wireless mouse")
pretty(response)

## Query 2 — Vague query → catalog-grounded clarification

The query is too vague to retrieve meaningful results. The system should NOT guess —
it should ask a clarifying question grounded in what the catalog actually returned.

In [ ]:
response = await router.handle(query="something for my dog")
pretty(response)

## Query 3 — Basket query, cold start

A multi-item intent. The intent agent flags `is_basket_query=True`, triggering the
query expander. The expander does a catalog pre-flight to discover available categories,
then proposes 3-5 sub-queries within those categories.

After expansion, each sub-query retrieves products with a relevance threshold filter —
sub-queries returning nothing above threshold are dropped silently. The dropped
sub-queries are surfaced in `debug_dropped_queries` for transparency.

Watch for:
- `is_basket_query=True` in the intent result
- The expansion proposed vs. dropped lists
- Mixed-partner basket (dm cleaning + Amazon storage/tools)
- Cold-start diversity bonus computed from result-set distribution

In [ ]:
response = await router.handle(query="essentials for moving into a new house")
pretty(response)

## Query 4 — Same query, with an EDEKA-heavy user

The same query as Query 3, but now with `user_id="user_edeka_heavy"`. The user
shops EDEKA 70% of the time, dm 20%, Amazon 10%.

The loyalty ranker uses the affinity profile to compute a diversity bonus that
*inversely* weights partners — boosting dm and Amazon products to surface
partners this user under-uses. PAYBACK's business model rewards diversification.

Compare to Query 3 — same intent, same expansion, same retrieval. The ranker
re-orders based on user signal.

In [ ]:
edeka_heavy = UserContext(
    user_id="user_edeka_heavy",
    partner_affinity={Partner.dm: 0.2, Partner.edeka: 0.7, Partner.amazon: 0.1},
    is_new_user=False,
)

response = await router.handle(
    query="essentials for moving into a new house",
    user_context=edeka_heavy,
)
pretty(response)

## Query 5 — German query with partner mention

`Windeln bei dm` ("diapers at dm" in German). This exercises three things at once:

- **Language detection**: should be `de`
- **Cross-lingual retrieval**: the embedding model is multilingual end-to-end —
  German queries retrieve from German product descriptions without translation
- **Partner mention extraction**: `target_partner=dm`, which constrains retrieval
  to dm's catalog only

Expectation: all results from dm, all genuine diaper products.

In [ ]:
response = await router.handle(query="Windeln bei dm")
pretty(response)

## Query 6 — Navigational without partner → polish branch

When the user expresses intent to navigate ("take me to the shop") but does NOT name
a partner, the system should ask which partner. This is a hardcoded fast-path —
no second LLM call, no retrieval — because the answer is fully determined.

Watch for:
- Latency dramatically lower than basket queries (only one LLM call)
- Hardcoded clarifying question + options

In [ ]:
response = await router.handle(query="take me to the shop")
pretty(response)

## Query 7 — Support intent → out-of-scope

Queries about points, accounts, or service get classified as `intent=support` and
short-circuited to a graceful out-of-scope response pointing at PAYBACK support.
Honest scope boundary rather than fake clarification.

In [ ]:
response = await router.handle(query="how do I redeem my points")
pretty(response)

## Quick verdict on what's built

After running the seven queries above, here's an honest summary of what works and
what's limited.

### What works

- **Synthetic catalog generation** with batched LLM tool-use, per-item Pydantic
  validation, and a per-batch acceptance threshold
- **Multilingual retrieval** with a regression test enforcing cross-lingual capability
- **Intent classification** with refined `target_partner` semantics — partner extraction
  on any recognized partner mention, null fallback for unrecognized partners (REWE, Lidl)
- **Catalog-grounded query expansion** with pre-flight category discovery and
  per-sub-query relevance filtering, with dropped-query transparency
- **Loyalty ranker** with three weighted signals (semantic + commercial + diversity)
  and cold-start fallback to result-set diversity
- **Router** with five branches including polished out-of-scope and partner-picker
  fast paths
- **Three-metric evaluation** — intent accuracy, retrieval precision@5/recall@5,
  end-to-end LLM-as-judge with Claude Opus + judge-vs-human agreement validation
- **Honest observability** — every layer reports its decisions, dropped queries,
  costs, and latencies in the response metadata

### What's limited

**Synthetic catalog coverage** (ADR-008): the ~700 synthetic products cover
dm/EDEKA/Amazon for their real-world categories, with limited cross-partner overlap.
Queries for sparse categories surface semantically-adjacent products rather than
literal matches (e.g. `Schokolade` returns chocolate-flavored yogurts and ice creams —
no chocolate bars in the catalog). The system is doing the right thing on what's there;
the fix is data, not code.

**Multilingual embedding model behavior** (ADR-010): the current model
(`paraphrase-multilingual-MiniLM-L12-v2`, 384-dim) handles simple, concrete bilingual
terms well (`Windeln`, `Bio Olivenöl`) but struggles on abstract compound German nouns
(`Grundausstattung`, `Heimtextilien`, `Aufbewahrung`). For such queries, pre-flight
retrieval returns weakly-matched items from adjacent categories. The honest fix is the
larger multilingual model (e.g. `paraphrase-multilingual-mpnet-base-v2` or Vertex AI's
`text-multilingual-embedding-002`), swappable via the `Embedder` interface.

**Single LLM provider for judge eval**: judge uses Claude Opus, system uses Claude
Sonnet. Same family, different size — reduces but doesn't eliminate self-preference
bias. Production would cross-evaluate against a different provider (e.g. Gemini).

## Evaluation results

Three metrics, all reproducible:

In [ ]:
import glob
import os

results_dirs = sorted(glob.glob("evals/results/*"))

print("Evaluation reports available:")
for d in results_dirs[-3:]:
    name = os.path.basename(d)
    print(f"  evals/results/{name}/report.md")

print("\nFor full results, open each report.md file.")
print("\nKey metrics to look for:")
print("  Intent:     overall accuracy, per-dimension breakdown, failure clusters")
print("  Retrieval:  mean precision@5, mean recall@5, weakest query class")
print("  E2E:        mean total score (X/9), per-dimension means, judge-human agreement")

## What would change in production

If this system were going to PAYBACK production, the priorities would be:

1. **Real partner catalogs** — replace synthetic data; expect retrieval quality to
   improve substantially when product descriptions reflect real merchandising
2. **Better multilingual embedding** — swap to a stronger model (768-dim+) for
   German abstract-noun handling
3. **Vertex AI Vector Search** — at PAYBACK scale (millions of products + users),
   colocate embeddings with the existing BigQuery footprint (see `bigquery_store.py`
   for the production stub)
4. **Real user profiles** — replace mock JSON profiles with a call to PAYBACK's
   user-profile service; the `UserContext` schema and ranker code stay unchanged
5. **Cross-provider judge eval** — add Gemini or GPT-4 as a second judge for
   bias-controlled evaluation; the abstract `LLMClient` interface makes this
   a single class addition
6. **Production observability** — wire the existing structured logging into
   Langfuse or LangSmith for prompt-version tracking, cost-per-feature dashboards,
   and regression alerts
7. **Rate limiting and auth** — add Cloud Armor + JWT validation at the API edge
8. **Load testing** — k6 / Locust against staging; tune Cloud Run concurrency
   and min-instances for the latency target

## Repository

- `app/` — production code
- `evals/` — three evaluation runners + labeled datasets
- `data/` — synthetic catalogs (~700 products) + mock user profiles
- `tests/` — pytest suite covering router, ranker, retrieval, intent, expansion
- `docs/decisions.md` — Architecture Decision Records (ADR-001 through ADR-010)
- `scripts/try_router.py`, `scripts/try_api.py` — manual smoke tests
- `README.md` — quick start, demo queries, architecture diagram